# 11 — Dependency Injection

## Objectives
- Understand Dependency Injection (DI) and its benefits
- Implement Constructor, Setter, and Interface injection
- Connect DI to SOLID's Dependency Inversion Principle
- Understand how Spring Framework uses DI

## What is DI?
Dependency Injection is a technique where objects receive their dependencies from external sources rather than creating them.
```
Without DI: Class creates its own dependencies (tight coupling)
With DI:    Dependencies are injected from outside (loose coupling)
```

In [2]:
// 1. First, define the missing concrete classes so the bad example can compile
class MySQLDatabase {
    public void save(String data) { 
        System.out.println("[MySQL] Saved: " + data); 
    }
}

class GmailEmailer {
    public void send(String to, String msg) { 
        System.out.println("[Gmail] Sent: " + msg); 
    }
}

// 2. NOW you can showcase the tightly coupled version without errors
class OrderServiceBad {
    // OrderService creates its own dependencies
    private MySQLDatabase db = new MySQLDatabase();     // hardcoded!
    private GmailEmailer emailer = new GmailEmailer();  // hardcoded!
    
    public void placeOrder(String orderId, String customerEmail) {
        db.save("Order: " + orderId);
        emailer.send(customerEmail, "Order " + orderId + " confirmed!");
    }
}

// --- VISUAL SEPARATION FOR YOUR DEMO ---
System.out.println("--- Running Loosely Coupled DI Example ---");

// 3. WITH DI (loosely coupled — GOOD)
interface Database { void save(String data); }
interface Emailer  { void send(String to, String msg); }

// We reuse or redefine the implementations for the interface strategy
class MySQLDatabaseDI implements Database {
    public void save(String data) { System.out.println("[MySQL-DI] Saved: " + data); }
}

class MockDatabase implements Database { // for testing!
    public void save(String data) { System.out.println("[MockDB] Saved: " + data); }
}

class SmtpEmailer implements Emailer {
    public void send(String to, String msg) { System.out.println("[SMTP] Email to " + to + ": " + msg); }
}

// OrderService with Constructor Injection
class OrderService {
    private final Database db;       // depends on abstraction
    private final Emailer emailer;   // depends on abstraction
    
    // Constructor Injection
    public OrderService(Database db, Emailer emailer) {
        this.db = db;
        this.emailer = emailer;
    }
    
    public void placeOrder(String orderId, String customerEmail) {
        db.save("Order: " + orderId);
        emailer.send(customerEmail, "Order " + orderId + " confirmed!");
        System.out.println("Order placed: " + orderId);
    }
}

// Production: inject real dependencies
OrderService prodService = new OrderService(new MySQLDatabaseDI(), new SmtpEmailer());
prodService.placeOrder("ORD-001", "customer@gmail.com");

System.out.println();

// Testing: inject mock dependencies (no real DB/email!)
OrderService testService = new OrderService(new MockDatabase(), (to, msg) -> System.out.println("[Mock] Email: " + msg));
testService.placeOrder("ORD-TEST", "test@test.com");

--- Running Loosely Coupled DI Example ---
[MySQL-DI] Saved: Order: ORD-001
[SMTP] Email to customer@gmail.com: Order ORD-001 confirmed!
Order placed: ORD-001

[MockDB] Saved: Order: ORD-TEST
[Mock] Email: Order ORD-TEST confirmed!
Order placed: ORD-TEST


## Spring DI Preview
```java
@Service
public class OrderService {
    @Autowired private Database db;     // Spring injects!
    @Autowired private Emailer emailer; // Spring injects!
}
```

## Mini Challenge
Refactor a `ReportService` that currently creates its own `CSVExporter` and `DatabaseLogger` to use constructor injection.

In [3]:
interface Exporter {
    void export(String data);
}

interface Logger {
    void log(String message);
}

In [4]:
class CSVExporter implements Exporter {
    public void export(String data) {
        System.out.println("Exporting data to CSV format: " + data);
    }
}

class DatabaseLogger implements Logger {
    public void log(String message) {
        System.out.println("[DB LOG]: " + message);
    }
}

In [5]:
class ReportService {
    private final Exporter exporter;
    private final Logger logger;

    // Constructor Injection: Dependencies are provided from the outside
    public ReportService(Exporter exporter, Logger logger) {
        this.exporter = exporter;
        this.logger = logger;
    }

    public void generateReport(String reportData) {
        logger.log("Generating report...");
        exporter.export(reportData);
        logger.log("Report generated successfully.");
    }
}

In [6]:
// Production Scenario: Injecting the real implementations
System.out.println("--- Running Production ---");
ReportService prodService = new ReportService(new CSVExporter(), new DatabaseLogger());
prodService.generateReport("Q2 Financial Summary");

System.out.println();

// Testing Scenario: Injecting mocks (No real file writing or DB connections!)
System.out.println("--- Running Unit Test ---");
Exporter mockExporter = data -> System.out.println("[Mock] Verified export called with: " + data);
Logger mockLogger = msg -> System.out.println("[Mock] Log captured: " + msg);

ReportService testService = new ReportService(mockExporter, mockLogger);
testService.generateReport("Test Data");

--- Running Production ---
[DB LOG]: Generating report...
Exporting data to CSV format: Q2 Financial Summary
[DB LOG]: Report generated successfully.

--- Running Unit Test ---
[Mock] Log captured: Generating report...
[Mock] Verified export called with: Test Data
[Mock] Log captured: Report generated successfully.
